In [1]:
!pip install catboost xgboost lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

X_test = test.drop(columns=['id'], axis=1)

train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Age                      630000 non-null  int64  
 2   Sex                      630000 non-null  int64  
 3   Chest pain type          630000 non-null  int64  
 4   BP                       630000 non-null  int64  
 5   Cholesterol              630000 non-null  int64  
 6   FBS over 120             630000 non-null  int64  
 7   EKG results              630000 non-null  int64  
 8   Max HR                   630000 non-null  int64  
 9   Exercise angina          630000 non-null  int64  
 10  ST depression            630000 non-null  float64
 11  Slope of ST              630000 non-null  int64  
 12  Number of vessels fluro  630000 non-null  int64  
 13  Thallium                 630000 non-null  int64  
 14  Hear

In [23]:
train.columns = train.columns.str.lower()
train.head()

,id,age,sex,chest pain type,bp,cholesterol,fbs over 120,ekg results,max hr,exercise angina,st depression,slope of st,number of vessels fluro,thallium,heart disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


In [24]:
X = train.drop(columns=['id', 'heart disease'], axis=1)
y = train['heart disease']

In [25]:
# Define which are categorical
categorical_set = {'chest pain type', 'slope of st', 'thallium'}

# Split into categorical and numeric
categorical_cols = [col for col in X.columns if col in categorical_set]
numeric_cols = [col for col in X.columns if col not in categorical_set]

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['chest pain type', 'slope of st', 'thallium']
Numeric: ['age', 'sex', 'bp', 'cholesterol', 'fbs over 120', 'ekg results', 'max hr', 'exercise angina', 'st depression', 'number of vessels fluro']


In [7]:
#train['number of vessels fluro'].unique()
#train['slope of st'].unique()
#train['slope of st'].value_counts(normalize=True)

## L2 Scaling, Encoding n k fold stratification

In [26]:
y = y.map({'Absence':0, 'Presence':1})

In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


model_set = {
    'Random Forest': RandomForestClassifier(
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        random_state=42
    ),
    'LightGBM': LGBMClassifier(
        random_state=42
    ),
    'CatBoost': CatBoostClassifier(
        random_state=42,
        verbose=0
    )
}

""""
'Logistic Regression': LogisticRegression(
        random_state=42,
        max_iter=1000
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=42
    ),
    'AdaBoost': AdaBoostClassifier(
        random_state=42
    )
"""

'"\n\'Logistic Regression\': LogisticRegression(\n        random_state=42,\n        max_iter=1000\n    ),\n    \'Gradient Boosting\': GradientBoostingClassifier(\n        random_state=42\n    ),\n    \'AdaBoost\': AdaBoostClassifier(\n        random_state=42\n    )\n'

In [10]:
results = {}

# store OOF predictions for blending
oof_predictions = {}

for model_name, model in model_set.items():

    print(f"\nModel: {model_name}")
    fold_scores = []

    # initialize OOF array for this model
    oof = np.zeros(len(X))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # fresh preprocessor each fold
        encoder = OneHotEncoder(drop='first', sparse_output=False)

        preprocessor = ColumnTransformer(
            transformers=[
                ('cat', encoder, categorical_cols),
                ('num', 'passthrough', numeric_cols)
            ]
        )

        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        val_probs = pipeline.predict_proba(X_val_fold)[:, 1]

        # store OOF predictions
        oof[val_idx] = val_probs

        fold_score = log_loss(y_val_fold, val_probs)
        fold_scores.append(fold_score)

        print(f"  Fold {fold+1}: {fold_score:.5f}")

    mean_score = np.mean(fold_scores)
    std_score = np.std(fold_scores)

    results[model_name] = (mean_score, std_score)

    # save OOF predictions for blending
    oof_predictions[model_name] = oof

    print(f"  Mean CV: {mean_score:.5f}")
    print(f"  Std CV:  {std_score:.5f}")



Model: Random Forest
  Fold 1: 0.37737
  Fold 2: 0.37958
  Fold 3: 0.37321
  Fold 4: 0.37857
  Fold 5: 0.36319
  Mean CV: 0.37438
  Std CV:  0.00600

Model: XGBoost
  Fold 1: 0.26896
  Fold 2: 0.27225
  Fold 3: 0.26940
  Fold 4: 0.27104
  Fold 5: 0.26856
  Mean CV: 0.27004
  Std CV:  0.00139

Model: LightGBM
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.045129 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 423
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 1: 0.26933
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.045163 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 418
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 2: 0.27206
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.067700 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 415
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383


...

  Fold 3: 0.26978
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.045424 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 417
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 4: 0.27104
[LightGBM] [Info] Number of positive: 225964, number of negative: 278036
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.045681 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 418
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448341 -> initscore=-0.207375
[LightGBM] [Info] Start training from score -0.207375


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  Fold 5: 0.26905
  Mean CV: 0.27025
  Std CV:  0.00113

Model: CatBoost
  Fold 1: 0.26772
  Fold 2: 0.27087
  Fold 3: 0.26806
  Fold 4: 0.26988
  Fold 5: 0.26694
  Mean CV: 0.26869
  Std CV:  0.00145


In [27]:
# Blend only top boosting models
blend_probs = (
    0.4 * oof_predictions["CatBoost"] +
    0.3 * oof_predictions["XGBoost"] +
    0.3 * oof_predictions["LightGBM"]
)

blend_logloss = log_loss(y, blend_probs)

print("\nBlended Model (Cat + XGB + LGB)")
print(f"Blended LogLoss: {blend_logloss:.5f}")



Blended Model (Cat + XGB + LGB)
Blended LogLoss: 0.26836


In [29]:
test_preds = {}

top_models = {
    "CatBoost": model_set["CatBoost"],
    "XGBoost": model_set["XGBoost"],
    "LightGBM": model_set["LightGBM"]
}

# Convert X_test column names to lowercase to match X
X_test.columns = X_test.columns.str.lower()

for model_name, model in top_models.items():

    print(f"\nTraining full model: {model_name}")

    encoder = OneHotEncoder(drop='first', sparse_output=False)

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', encoder, categorical_cols),
            ('num', 'passthrough', numeric_cols)
        ]
    )

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Train on full training data
    pipeline.fit(X, y)

    # Predict probabilities on test data
    test_probs = pipeline.predict_proba(X_test)[:, 1]

    test_preds[model_name] = test_probs


Training full model: CatBoost

Training full model: XGBoost

Training full model: LightGBM
[LightGBM] [Info] Number of positive: 282454, number of negative: 347546
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.056302 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 418
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448340 -> initscore=-0.207381
[LightGBM] [Info] Start training from score -0.207381


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [30]:
blend_test_probs = (
    0.4 * test_preds["CatBoost"] +
    0.3 * test_preds["XGBoost"] +
    0.3 * test_preds["LightGBM"]
)


In [31]:
submission = pd.DataFrame({
    "id": test["id"],
    "heart disease": blend_test_probs
})

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")


Submission file created successfully!
